In [0]:
%sql
-- LEADS_ACTIVITY_FACT — one row per activity (latest version), attributed to real staff via staff_dim
CREATE OR REPLACE TABLE crm_ingestion.gold.leads_activity_fact AS (
    SELECT
        a.id AS activity_id,
        a.lead_id,
        a.type,
        a.activity_at,
        a.direction,
        a.status,
        a.disposition,
        a.subject,
        a.body_preview,
        a.text,
        a.starts_at,
        a.ends_at,
        a.outcome_id,
        a.custom_activity_type_id,
        a.created_by AS csm_id,
        COALESCE(u.csm_name, a.created_by_name) AS created_by_name,
        u.csm_email,
        a.date_created,
        a.date_updated,
        a.note,
        a.contact_id,
        a.organization_id,
        a.bronze_insert_date
    FROM (
        SELECT
            id,
            lead_id,
            type,
            activity_at,
            direction,
            status,
            disposition,
            subject,
            body_preview,
            text,
            starts_at,
            ends_at,
            outcome_id,
            custom_activity_type_id,
            created_by,
            created_by_name,
            date_created,
            date_updated,
            note,
            contact_id,
            organization_id,
            bronze_insert_date,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.lead_activites_processed
    ) a
    LEFT JOIN
        crm_ingestion.gold.staff_dim u
    ON
        a.created_by = u.csm_id
    WHERE
        a.id IS NOT NULL
    AND
        a.rn = 1
)
    

In [0]:
%sql
SELECT *
FROM crm_ingestion.gold.leads_activity_fact
LIMIT 100